In [ ]:
#1-Acesse o drive---

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive montado em /content/drive")

Mounted at /content/drive
✅ Drive montado em /content/drive


In [ ]:
#2-Importar as bibliotecas e bases.

In [ ]:
!pip install -q ultralytics

import os, glob, time, random, shutil
from pathlib import Path
import torch
import torch.nn as nn
from torchvision import models
from ultralytics import YOLO


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 66.7 MB/s eta 0:00:00


In [ ]:
#2-Treinamento yolo cbdiz-parou aqui

In [ ]:
#Copiar a base

import shutil
from pathlib import Path

src = Path("/content/drive/MyDrive/replicacao_article/inbreast2")
dst = Path("/content/inbreast")

shutil.copytree(src, dst)


In [ ]:

#Ajuste os caminhos
NEW_ROOT = f"/content/dataset_teacher_student_v2"

# NOVOS subdiretórios
NEW_IMG_TRAIN = f"{NEW_ROOT}/images/train"
NEW_LBL_TRAIN = f"{NEW_ROOT}/labels/train"
NEW_IMG_VAL   = f"{NEW_ROOT}/images/val"
NEW_LBL_VAL   = f"{NEW_ROOT}/labels/val"
NEW_IMG_TEST  = f"{NEW_ROOT}/images/test"
NEW_LBL_TEST  = f"{NEW_ROOT}/labels/test"

In [ ]:

#Criação do data.yaml
import glob
import os
import shutil

DATA_YAML = f"""
path: {NEW_ROOT}
train: {NEW_IMG_TRAIN}
val:   {NEW_IMG_VAL}
test:  {NEW_IMG_TEST}
names: ["Mass"]   # ajuste suas classes
nc: 1
"""
with open(f"{NEW_ROOT}/data.yaml", "w") as f:
    f.write(DATA_YAML.strip()+"\n")

HYP_YAML = """
# Geometria
degrees: 5.0
translate: 0.10
scale: 0.50
shear: 2.0
perspective: 0.0005
# Flips
flipud: 0.0
fliplr: 0.5
# Composições
mosaic: 1.0
mixup: 0.10
copy_paste: 0.0
# Cores (HSV)
hsv_h: 0.015
hsv_s: 0.70
hsv_v: 0.40
"""
with open(f"{NEW_ROOT}/hyp.yaml", "w") as f:
    f.write(HYP_YAML.strip()+"\n")

# limpar possíveis caches (no NEW_ROOT geralmente não tem, mas por garantia)
for c in glob.glob(f"{NEW_ROOT}/labels/*/*.cache"):
    try: os.remove(c)
    except: pass

print("✔ data.yaml e hyp.yaml criados em:", NEW_ROOT)

✔ data.yaml e hyp.yaml criados em: /content/dataset_teacher_student_v2


In [ ]:
#2.1 Alteração de backbone fica a sua escolha, pode escolher o resnet ou o densenet

In [ ]:

#Alteração de backbone
import torch.nn as nn
from torchvision import models

class CustomResNet152Backbone(nn.Module):
    """
    Saídas compatíveis com a neck do YOLOv8:
      P3 (stride~8)  -> 128 canais
      P4 (stride~16) -> 256 canais
      P5 (stride~32) -> 512 canais
    """
    def __init__(self, out_channels=(128, 256, 512), pretrained=True):
        super().__init__()
        try:
            weights = models.ResNet152_Weights.IMAGENET1K_V2 if pretrained else None
            resnet = models.resnet152(weights=weights)
        except AttributeError:
            resnet = models.resnet152(pretrained=pretrained)

        self.stem   = nn.Sequential(resnet.conv1, resnet.bn1, nn.ReLU(inplace=True), resnet.maxpool)  # stride 4
        self.layer1 = resnet.layer1  # stride 4
        self.layer2 = resnet.layer2  # stride 8
        self.layer3 = resnet.layer3  # stride 16
        self.layer4 = resnet.layer4  # stride 32

        self.red2 = nn.Sequential(nn.Conv2d( 512, out_channels[0], 1, bias=False),
                                  nn.BatchNorm2d(out_channels[0]), nn.SiLU(inplace=True))
        self.red3 = nn.Sequential(nn.Conv2d(1024, out_channels[1], 1, bias=False),
                                  nn.BatchNorm2d(out_channels[1]), nn.SiLU(inplace=True))
        self.red4 = nn.Sequential(nn.Conv2d(2048, out_channels[2], 1, bias=False),
                                  nn.BatchNorm2d(out_channels[2]), nn.SiLU(inplace=True))

    def forward(self, x):
        x  = self.stem(x)
        x1 = self.layer1(x)      # stride 4
        c3 = self.layer2(x1)     # stride 8
        c4 = self.layer3(c3)     # stride 16
        c5 = self.layer4(c4)     # stride 32
        P3 = self.red2(c3)       # 128
        P4 = self.red3(c4)       # 256
        P5 = self.red4(c5)       # 512
        return [P3, P4, P5]

In [ ]:
#Alteracao de backbone

import torch
import torch.nn as nn
from torchvision import models


class CustomDenseNet201Backbone(nn.Module):
    """
    DenseNet-201 backbone custom para YOLOv8/YOLO11.
    Saídas:
      P3 (stride ~8)  -> out_channels[0] (default: 128)
      P4 (stride ~16) -> out_channels[1] (default: 256)
      P5 (stride ~32) -> out_channels[2] (default: 512)
    """
    def __init__(self, out_channels=(128, 256, 512), pretrained=True):
        super().__init__()

        # Carregar DenseNet201
        try:
            weights = models.DenseNet201_Weights.IMAGENET1K_V1 if pretrained else None
            dnet = models.densenet201(weights=weights).features
        except AttributeError:
            dnet = models.densenet201(pretrained=pretrained).features

        # Stem: conv0 + norm0 + relu0 + pool0  (stride 4)
        self.stem = nn.Sequential(
            dnet.conv0,
            dnet.norm0,
            nn.ReLU(inplace=True),
            dnet.pool0,
        )

        # Blocos e transições
        self.block1 = dnet.denseblock1
        self.trans1 = dnet.transition1   # stride ~8

        self.block2 = dnet.denseblock2
        self.trans2 = dnet.transition2   # stride ~16

        self.block3 = dnet.denseblock3
        self.trans3 = dnet.transition3   # stride ~32

        # Pegar os canais reais de saída de cada transition
        c3_in = self.trans1.conv.out_channels  # canais em P3
        c4_in = self.trans2.conv.out_channels  # canais em P4
        c5_in = self.trans3.conv.out_channels  # canais em P5  (no seu caso: 896)

        # Cabeças de redução de canais
        self.red3 = nn.Sequential(
            nn.Conv2d(c3_in, out_channels[0], kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels[0]),
            nn.SiLU(inplace=True),
        )

        self.red4 = nn.Sequential(
            nn.Conv2d(c4_in, out_channels[1], kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels[1]),
            nn.SiLU(inplace=True),
        )

        self.red5 = nn.Sequential(
            nn.Conv2d(c5_in, out_channels[2], kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels[2]),
            nn.SiLU(inplace=True),
        )

    def forward(self, x):
        x = self.stem(x)        # stride 4

        x = self.block1(x)
        c3 = self.trans1(x)     # stride ~8

        x = self.block2(c3)
        c4 = self.trans2(x)     # stride ~16

        x = self.block3(c4)
        c5 = self.trans3(x)     # stride ~32

        P3 = self.red3(c3)      # [B, 128, H/8,  W/8 ]
        P4 = self.red4(c4)      # [B, 256, H/16, W/16]
        P5 = self.red5(c5)      # [B, 512, H/32, W/32]

        return [P3, P4, P5]

In [ ]:
#2.2 Importa o modelo

In [ ]:
#Importar o modelo alterado

import torch
from ultralytics import YOLO

model = YOLO("yolov8s.pt")
model.model.backbone = CustomResNet152Backbone(pretrained=True)

# sanity check (deve imprimir ~[(1,128,80,80),(1,256,40,40),(1,512,20,20)])
with torch.no_grad():
    feats = model.model.backbone(torch.randn(1,3,640,640))
    print("Shapes das 3 features:", [tuple(f.shape) for f in feats])


Shapes das 3 features: [(1, 128, 80, 80), (1, 256, 40, 40), (1, 512, 20, 20)]


In [ ]:
results = model.train(
    data=f"{NEW_ROOT}/data.yaml",
    epochs=50,
    batch=16,
    imgsz=640,
    name="yolov8_resnet152_newdirs",
    close_mosaic=10,     # fecha mosaic nas 10 últimas épocas
    workers=2,
    plots=True,

    save_period=5,
    optimizer="AdamW", lr0=1e-4,   # (opcional) descomente se quiser
)

Ultralytics 8.3.233 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_teacher_student_v2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8_resnet152_newdirs2, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=

In [ ]:
 #3----treinamento finetuning------

In [ ]:
#Importe o modelo do inbreast modificado  para por no code abaixo- vai-se realizar o kfold aqui com o cbdiz

In [ ]:
#com a base  importada execute isso

# Se ainda não tiver instalado
!pip install -q ultralytics scikit-learn

import os
import shutil
from sklearn.model_selection import KFold

from ultralytics import YOLO


# =====================================================
# CONFIGURAÇÕES DO USUÁRIO
# =====================================================

# 1) Modelo pré-treinado no VinDr (o que deu mAP 63.5)
PRETRAINED_MODEL = "/content/drive/MyDrive/replicacao_article/replicacao_tudo_results/runs/detect/yolov8_resnet152_newdirs/weights/best.pt"
#    ↑ TROQUE para o caminho do SEU best.pt

# 2) Diretórios do INBreast em formato YOLO (1 classe mass)
IMAGES_DIR = "/content/inbreast_fcagado/images"   # ajuste se for /content/inbreast2/images etc.
LABELS_DIR = "/content/inbreast_fcagado/labels"   # ajuste se for /content/inbreast2/labels etc.

# 3) Pasta para salvar tudo
OUTPUT_DIR = "/content/anjo_7_fold"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =====================================================
# LISTAR IMAGENS QUE TÊM LABEL
# =====================================================

valid_exts = (".png", ".jpg", ".jpeg")

image_files = [
    f for f in os.listdir(IMAGES_DIR)
    if f.lower().endswith(valid_exts)
    and os.path.exists(os.path.join(LABELS_DIR, os.path.splitext(f)[0] + ".txt"))
]

print(f"Imagens INBreast com label encontradas: {len(image_files)}")

# =====================================================
# DEFINIR AUGMENT DO FINE-TUNING (VINDr → INBreast)
# medium modificado do artigo:
# flipud = 0.5, degrees = 10, shear = 10, scale = 0.9,
# mosaic/mixup/copy_paste = 0, translate = 0.1
# =====================================================

augment_ft = dict(
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,
    degrees=10.0,
    shear=10.0,
    scale=0.9,
    translate=0.1,
    flipud=0.5,
    fliplr=0.5,
    perspective=0.0,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
)

# =====================================================
# CONFIG TREINO
# =====================================================

NUM_FOLDS = 5
IMG_SIZE  = 640
EPOCHS    = 50
BATCH     = 8

kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)
fold_maps = []

# =====================================================
# LOOP DOS 5 FOLDS
# =====================================================

for fold, (train_idx, val_idx) in enumerate(kf.split(image_files), 1):
    print(f"\n==============================")
    print(f"          FOLD {fold}/{NUM_FOLDS}")
    print(f"==============================\n")

    fold_dir   = os.path.join(OUTPUT_DIR, f"fold_{fold}")
    img_train  = os.path.join(fold_dir, "images/train")
    img_val    = os.path.join(fold_dir, "images/val")
    lbl_train  = os.path.join(fold_dir, "labels/train")
    lbl_val    = os.path.join(fold_dir, "labels/val")

    for p in [img_train, img_val, lbl_train, lbl_val]:
        os.makedirs(p, exist_ok=True)

    # Função pra copiar imagem + label
    def copy_pairs(files, out_img, out_lbl):
        for fname in files:
            base = os.path.splitext(fname)[0]
            img_src = os.path.join(IMAGES_DIR, fname)
            lbl_src = os.path.join(LABELS_DIR, base + ".txt")

            shutil.copy2(img_src, os.path.join(out_img, fname))
            shutil.copy2(lbl_src, os.path.join(out_lbl, base + ".txt"))

    # Listas de treino e validação deste fold
    train_files = [image_files[i] for i in train_idx]
    val_files   = [image_files[i] for i in val_idx]

    copy_pairs(train_files, img_train, lbl_train)
    copy_pairs(val_files,   img_val,   lbl_val)

    print(f"Treino: {len(train_files)} imagens | Validação: {len(val_files)} imagens")

    # =====================================================
    # CRIAR YAML DO FOLD (1 CLASSE: mass)
    # =====================================================

    yaml_path = os.path.join(fold_dir, "inbreast_mass.yaml")
    with open(yaml_path, "w") as f:
        f.write(f"""
path: {fold_dir}

train: images/train
val: images/val

names:
  0: mass
""")

    # =====================================================
    # TREINO DO FINE-TUNING PARA ESTE FOLD
    # =====================================================

    print(f"\n🔄 Treinando fine-tuning no FOLD {fold}...\n")

    # Carrega o modelo pré-treinado no VinDr
    model = YOLO(PRETRAINED_MODEL)

    results = model.train(
        data=yaml_path,
        imgsz=IMG_SIZE,
        epochs=EPOCHS,
        batch=BATCH,
        optimizer="AdamW",
        lr0=1e-4,
        device=0,
        project=os.path.join(OUTPUT_DIR, "results"),
        name=f"fold_{fold}",
        pretrained=True,   # usa os pesos do VinDr como base
        verbose=True,
        **augment_ft       # AUGMENT IGUAL AO PAPER (VINDr→INBreast)
    )

    # Tentar pegar mAP50 automático
    fold_map = None
    try:
        # Em algumas versões, métricas ficam em model.metrics ou results
        if hasattr(model, "metrics") and model.metrics is not None:
            # YOLOv8 geralmente usa 'metrics' com chaves tipo 'metrics/mAP50(B)'
            m = model.metrics
            fold_map = m.get("metrics/mAP50(B)", None) or m.get("metrics/mAP50-95(B)", None)
        elif hasattr(results, "metrics") and results.metrics is not None:
            m = results.metrics
            fold_map = m.get("metrics/mAP50(B)", None) or m.get("metrics/mAP50-95(B)", None)
    except Exception as e:
        print("Não foi possível ler o mAP automaticamente:", e)

    if fold_map is not None:
        fold_maps.append(fold_map)
        print(f"\n📌 FOLD {fold} – mAP (aprox): {fold_map}\n")
    else:
        print("\n⚠ mAP não foi capturado via código. Veja o summary no terminal ou o 'results.csv' desse fold.\n")

# =====================================================
# RESUMO FINAL DOS 5 FOLDS
# =====================================================

print("\n==============================")
print("     RESUMO FINAL - 5 FOLDS")
print("==============================\n")

if fold_maps:
    for i, v in enumerate(fold_maps, 1):
        print(f"Fold {i}: mAP ≈ {v}")
    print(f"\n🔵 MÉDIA DOS mAPs ≈ {sum(fold_maps)/len(fold_maps):.4f}")
else:
    print("mAP não foi extraído via código, mas todos os treinos foram realizados.")
    print("Confira em:", os.path.join(OUTPUT_DIR, "results"))

Imagens INBreast com label encontradas: 60

          FOLD 1/5

Treino: 48 imagens | Validação: 12 imagens

🔄 Treinando fine-tuning no FOLD 1...

Ultralytics 8.3.241 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA L4, 22693MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/anjo_7_fold/fold_1/inbreast_mass.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/replicacao_arti

In [ ]:
4-#Repita o passo 3 com o vindr alterado.